# Milestone 2: Differential Diagnosis Triage System
**Dataset, preprocessing, and running pre-existing code**

| Part | What this notebook shows |
|---|---|
| 1 | DDXPlus loaded and explored |
| 2 | Preprocessing: filter to chest-pain conditions, build evidence table |
| 3 | Pre-existing pretrained model: scispaCy (+ negation) extracting symptoms from text |
| 4 | Pre-existing library: pgmpy Bayesian Network running a query |
| 5 | Expected Information Gain on the toy network |

**Transfer learning:** `en_core_sci_md` was pretrained by AllenAI on biomedical text. We reuse it without retraining and add our own negation, synonym and mapping layers on top.

**Run order:** Run Part 0 first (install), then Runtime > Restart session if Colab asks, then run everything top to bottom.

## Part 0: Install pre-existing libraries

In [1]:
!python --version

Python 3.11.13


In [4]:
%pip install -q \
    "numpy==1.26.4" \
    "scipy==1.10.1" \
    "spacy==3.7.5" \
    "thinc==8.2.5" \
    "scispacy==0.5.4" \
    "negspacy==1.0.4" \
    "pgmpy==0.1.26" \
    plotly pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.9/58.9 kB 4.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 70.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.1/34.1 MB 33.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 83.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 920.2/920.2 kB 37.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 81.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 82.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 49.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.1/71.1 kB 7.4 MB/s eta 0:00:00


In [2]:
%pip install -q pandas numpy pgmpy plotly
%pip install -q "spacy>=3.7,<3.8" "scispacy==0.5.4"
%pip install -q negspacy
%pip install -q https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_core_sci_md-0.5.4.tar.gz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 49.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 14.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.9/58.9 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 105.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 64.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.1/34.1 MB 32.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 93.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 920.2/920.2 kB 54.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.1/71.1 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 116.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.4/31

In [1]:
import numpy, scipy, spacy, scispacy, negspacy, pgmpy

print("NumPy:", numpy.__version__)
print("SciPy:", scipy.__version__)
print("spaCy:", spacy.__version__)
print("scispaCy: OK")
print("negspacy: OK")
print("pgmpy:", pgmpy.__version__)

NumPy: 1.26.4
SciPy: 1.10.1
spaCy: 3.7.5
scispaCy: OK
negspacy: OK
pgmpy: 0.1.26


## Part 1: Dataset (DDXPlus)
Files needed from the DDXPlus figshare page: `release_conditions.json`, `release_evidences.json`, and **one** patients file.
For tonight use `release_validate_patients` (smaller) instead of the train file. The code accepts .csv or .zip.

Upload with the cell below, or mount Google Drive and set the paths yourself.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os

DATA_DIR = "/content/drive/MyDrive/DDXPlus"

print("Folder exists:", os.path.isdir(DATA_DIR))

if os.path.isdir(DATA_DIR):
    print("Files found:", os.listdir(DATA_DIR))

Folder exists: True
Files found: ['release_conditions.json', 'release_evidences.json', 'release_train_patients.zip']


In [3]:
import zipfile
import os

zip_path = "/content/drive/MyDrive/DDXPlus/release_train_patients.zip"

with zipfile.ZipFile(zip_path, "r") as z:
    for name in z.namelist():
        print(name)

release_train_patients


In [5]:
import zipfile

zip_path = "/content/drive/MyDrive/DDXPlus/release_train_patients.zip"

with zipfile.ZipFile(zip_path, "r") as z:
    with z.open("release_train_patients") as f:
        print(f.read(300).decode("utf-8", errors="replace"))

AGE,DIFFERENTIAL_DIAGNOSIS,SEX,PATHOLOGY,EVIDENCES,INITIAL_EVIDENCE
18,"[['Bronchitis', 0.19171203430383882], ['Pneumonia', 0.17579340398940366], ['URTI', 0.1607809719801254], ['Bronchiectasis', 0.12429044460990353], ['Tuberculosis', 0.11367177304035844], ['Influenza', 0.11057936110639896], ['HIV (i


In [7]:
import json, ast, zipfile
import pandas as pd
import numpy as np

base_path = "/content/drive/MyDrive/DDXPlus"

patients_file = f"{base_path}/release_train_patients.zip"

with zipfile.ZipFile(patients_file, "r") as z:
    with z.open("release_train_patients") as f:
        df = pd.read_csv(f)

with open(f"{base_path}/release_conditions.json") as f:
    conditions = json.load(f)

with open(f"{base_path}/release_evidences.json") as f:
    evidences = json.load(f)

print("Patients file:", patients_file)
print("Shape:", df.shape)
print("Columns:", list(df.columns))
print("Number of pathologies:", df["PATHOLOGY"].nunique())
print("Number of evidence codes:", len(evidences))

display(df.head())

Patients file: /content/drive/MyDrive/DDXPlus/release_train_patients.zip
Shape: (1025602, 6)
Columns: ['AGE', 'DIFFERENTIAL_DIAGNOSIS', 'SEX', 'PATHOLOGY', 'EVIDENCES', 'INITIAL_EVIDENCE']
Number of pathologies: 49
Number of evidence codes: 223


,AGE,DIFFERENTIAL_DIAGNOSIS,SEX,PATHOLOGY,EVIDENCES,INITIAL_EVIDENCE
0,18,"[['Bronchitis', 0.19171203430383882], ['Pneumo...",M,URTI,"['E_48', 'E_50', 'E_53', 'E_54_@_V_161', 'E_54...",E_91
1,21,"[['HIV (initial infection)', 0.518950056440760...",M,HIV (initial infection),"['E_9', 'E_27', 'E_50', 'E_51', 'E_53', 'E_54_...",E_50
2,19,"[['Bronchitis', 0.11278064619119596], ['Pneumo...",F,Pneumonia,"['E_53', 'E_54_@_V_179', 'E_54_@_V_192', 'E_55...",E_77
3,34,"[['URTI', 0.23859396799565236], ['Cluster head...",F,URTI,"['E_48', 'E_53', 'E_54_@_V_183', 'E_55_@_V_89'...",E_53
4,36,"[['URTI', 0.23677812769175735], ['Influenza', ...",M,URTI,"['E_49', 'E_50', 'E_53', 'E_54_@_V_183', 'E_55...",E_201


In [8]:
# All pathology names in the dataset (check which ones match our 4 conditions)
print(sorted(conditions.keys()))

['Acute COPD exacerbation / infection', 'Acute dystonic reactions', 'Acute laryngitis', 'Acute otitis media', 'Acute pulmonary edema', 'Acute rhinosinusitis', 'Allergic sinusitis', 'Anaphylaxis', 'Anemia', 'Atrial fibrillation', 'Boerhaave', 'Bronchiectasis', 'Bronchiolitis', 'Bronchitis', 'Bronchospasm / acute asthma exacerbation', 'Chagas', 'Chronic rhinosinusitis', 'Cluster headache', 'Croup', 'Ebola', 'Epiglottitis', 'GERD', 'Guillain-Barré syndrome', 'HIV (initial infection)', 'Influenza', 'Inguinal hernia', 'Larygospasm', 'Localized edema', 'Myasthenia gravis', 'Myocarditis', 'PSVT', 'Pancreatic neoplasm', 'Panic attack', 'Pericarditis', 'Pneumonia', 'Possible NSTEMI / STEMI', 'Pulmonary embolism', 'Pulmonary neoplasm', 'SLE', 'Sarcoidosis', 'Scombroid food poisoning', 'Spontaneous pneumothorax', 'Spontaneous rib fracture', 'Stable angina', 'Tuberculosis', 'URTI', 'Unstable angina', 'Viral pharyngitis', 'Whooping cough']


In [9]:
# Look for a musculoskeletal-type condition
print([c for c in conditions if any(k in c.lower() for k in ['musc','costo','chest wall','rib'])])

['Spontaneous rib fracture']


## Part 2: Preprocessing
Steps: (1) map DDXPlus pathologies to our target conditions, (2) filter, (3) parse the evidence list into a binary table, (4) turn codes into readable names.

**Edit `CONDITION_MAP` after looking at the two cells above.** If there is no musculoskeletal condition, use Pericarditis as the 4th condition and say so in the viva.

In [10]:
CONDITION_MAP = {
    'Unstable angina': 'ACS',
    'Stable angina': 'ACS',
    'Possible NSTEMI / STEMI': 'ACS',
    'GERD': 'GERD',
    'Panic attack': 'Panic',
    'Pericarditis': 'Pericarditis',   # replace/remove depending on what exists
}
missing = [k for k in CONDITION_MAP if k not in conditions]
print("Names not found in dataset (fix these):", missing)

print("Before filtering:", len(df))
sub = df[df['PATHOLOGY'].isin(CONDITION_MAP)].copy()
sub['CONDITION'] = sub['PATHOLOGY'].map(CONDITION_MAP)
print("After filtering:", len(sub))
print(sub['CONDITION'].value_counts())

Names not found in dataset (fix these): []
Before filtering: 1025602
After filtering: 133282
CONDITION
ACS             59499
GERD            25979
Panic           25019
Pericarditis    22785
Name: count, dtype: int64


In [11]:
# Parse EVIDENCES (stored as a string like "['E_91', 'E_53_@_V_12']")
def base_codes(s):
    items = ast.literal_eval(s)
    return {i.split('_@_')[0] for i in items}

sub['EV_SET'] = sub['EVIDENCES'].apply(base_codes)

# Binary table: 1 = evidence reported, 0 = not reported
from sklearn.preprocessing import MultiLabelBinarizer
mlb = MultiLabelBinarizer()
X = pd.DataFrame(mlb.fit_transform(sub['EV_SET']), columns=mlb.classes_, index=sub.index)

# Readable column names
name = {c: evidences[c]['question_en'] for c in X.columns}
print("Evidence columns:", X.shape[1])
print("Missing values:", int(sub['EVIDENCES'].isna().sum()))
print("Note: in DDXPlus, evidence not in the list = not reported/absent.")

Evidence columns: 54
Missing values: 0
Note: in DDXPlus, evidence not in the list = not reported/absent.


In [12]:
# Top evidences per condition: P(evidence | condition)
data = X.copy()
data['CONDITION'] = sub['CONDITION']
freq = data.groupby('CONDITION').mean().T
freq['overall'] = X.mean()
top = freq.sort_values('overall', ascending=False).head(25)
top.index = [name[c][:70] for c in top.index]
top.round(2)

CONDITION,ACS,GERD,Panic,Pericarditis,overall
Do you feel pain somewhere?,1.00,1.00,1.00,1.00,1.00
How fast did the pain appear?,1.00,1.00,1.00,1.00,1.00
How intense is the pain?,1.00,1.00,1.00,1.00,1.00
Characterize your pain:,1.00,1.00,1.00,1.00,1.00
Does the pain radiate to another location?,1.00,1.00,1.00,1.00,1.00
Have you traveled out of the country in the last 4 weeks?,1.00,1.00,1.00,1.00,1.00
How precisely is the pain located?,1.00,1.00,1.00,1.00,1.00
"Do you have pain somewhere, related to your reason for consulting?",1.00,0.99,1.00,1.00,1.00
Are you experiencing shortness of breath or difficulty breathing in a,0.64,0.00,0.71,0.63,0.53
Do you smoke cigarettes?,0.72,0.67,0.00,0.00,0.45


In [13]:
# Candidate nodes for our Bayesian Network: evidences that differ MOST between conditions
spread = (freq.drop(columns='overall').max(axis=1) - freq.drop(columns='overall').min(axis=1))
cand = spread.sort_values(ascending=False).head(20)
pd.DataFrame({'question': [name[c] for c in cand.index], 'spread': cand.values.round(2)}, index=cand.index)

,question,spread
E_3,Have you ever had a pericarditis?,0.84
E_157,"Have you recently had numbness, loss of sensat...",0.80
E_105,Have you ever had a heart attack or do you hav...,0.78
E_111,Do you feel like you are dying or were you afr...,0.77
E_69,Do you have diabetes?,0.76
E_81,Do you suffer from chronic anxiety?,0.75
E_0,Have you recently had a viral infection?,0.75
E_173,Do you have a burning sensation that starts in...,0.75
E_215,Do you have symptoms that get worse after eating?,0.74
E_80,Have you ever been diagnosed with depression?,0.73


## Part 3: Pre-existing pretrained NLP model (transfer learning)
`en_core_sci_md` is a pretrained biomedical model. Pipeline: **scispaCy NER -> our synonym matcher -> negation detection -> structured output.**

In [15]:
%pip install -q https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_core_sci_md-0.5.4.tar.gz

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.1/119.1 MB 9.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [16]:
import spacy

nlp = spacy.load("en_core_sci_md")
print("Model loaded successfully!")
print("Pipeline:", nlp.pipe_names)

Model loaded successfully!
Pipeline: ['tok2vec', 'tagger', 'attribute_ruler', 'lemmatizer', 'parser', 'ner']


/usr/local/lib/python3.11/dist-packages/spacy/language.py:2195: FutureWarning: Possible set union at position 6328
  deserializers["tokenizer"] = lambda p: self.tokenizer.from_disk(  # type: ignore[union-attr]


In [33]:
import spacy
from spacy.language import Language
from spacy.matcher import PhraseMatcher
from spacy.util import filter_spans
from spacy.tokens import Span
from negspacy.negation import Negex

nlp = spacy.load('en_core_sci_md')

# Our own synonym dictionary: canonical node -> phrasings
SYNONYMS = {
    'breathlessness': ['shortness of breath', 'breathless', 'breathing difficulty', "can't catch my breath", 'difficulty breathing', 'dyspnea'],
    'chest_pressure': ['chest pressure', 'heavy pressure in the centre of his chest', 'chest tightness', 'tight chest', 'pressure in the chest', 'heaviness in the chest'],
    'chest_pain': ['chest pain', 'pain in the chest'],
    'sweating': ['sweating', 'sweaty', 'diaphoresis', 'cold sweat'],
    'nausea': ['nausea', 'nauseous', 'feel sick', 'queasy'],
    'radiation_arm_jaw': ['pain radiating to the arm', 'pain radiating to the jaw', 'radiates to the left arm', 'pain in the left arm'],
    'heartburn': ['heartburn', 'burning sensation in the chest', 'acid reflux', 'burning in the chest'],
    'palpitations': ['palpitations', 'racing heart', 'heart racing'],
    'anxiety': ['anxiety', 'anxious', 'panic attack', 'panic'],
}
matcher = PhraseMatcher(nlp.vocab, attr='LOWER')
for node, phrases in SYNONYMS.items():
    matcher.add(node, [nlp.make_doc(p) for p in phrases])

@Language.component('synonym_ents')
def synonym_ents(doc):
    spans = []
    for match_id, start, end in matcher(doc):
        label = nlp.vocab.strings[match_id]
        spans.append(Span(doc, start, end, label=label))

    # Keep pretrained entities only when they don't overlap
    # with a custom synonym match.
    custom_spans = filter_spans(spans)
    custom_ranges = {(s.start, s.end) for s in custom_spans}

    other_ents = [
        ent for ent in doc.ents
        if not any(
            ent.start < end and start < ent.end
            for start, end in custom_ranges
        )
    ]

    doc.ents = filter_spans(other_ents + custom_spans)
    return doc

nlp.add_pipe('synonym_ents', after='ner')
nlp.add_pipe(
    'negex',
    config={'ent_types': [], 'chunk_prefix': ['no', 'denies', 'denied', 'without']},
    last=True
)
print(nlp.pipe_names)

['tok2vec', 'tagger', 'attribute_ruler', 'lemmatizer', 'parser', 'ner', 'synonym_ents', 'negex']


In [38]:
import re

NODES = set(SYNONYMS)

def extract(text):
    out = {}
    doc = nlp(text)

    for sent in doc.sents:
        sent_text = sent.text
        sent_doc = nlp(sent_text)

        for e in sent_doc.ents:
            if e.label_ not in NODES:
                continue

            phrase = e.text.lower()
            low_sent = sent_text.lower()
            pos = low_sent.find(phrase)

            # "Can't catch her breath" expresses breathlessness.
            if phrase in ("can't catch her breath", "can't catch my breath"):
                out[e.label_] = True
                continue

            # Detect common negation cues immediately before a symptom.
            prefix = low_sent[:pos] if pos >= 0 else ""
            prefix = re.split(r"[,;.!?]", prefix)[-1].strip()

            negated = bool(re.search(
                r"\b(no|not|never|denies|denied|without)\b",
                prefix
            ))

            # A contrast such as "but reports" ends the earlier negation.
            if re.search(r"\bbut\b.*\b(reports?|has|have|feels?)\b", prefix):
                negated = False

            out[e.label_] = not negated

    # Simple extras via regex
    m = re.search(r'(\d{2})[- ]year', text)
    if m:
        out['age'] = int(m.group(1))

    low = text.lower()
    if 'smoker' in low or 'smokes' in low:
        out['smoking'] = True
    if re.search(r'\b(male|man)\b', low):
        out['sex'] = 'M'
    if re.search(r'\b(female|woman)\b', low):
        out['sex'] = 'F'

    d = re.search(r'(\d+)\s*(minutes|min)', low)
    if d:
        out['duration_min'] = int(d.group(1))

    if 'walking' in low or 'exertion' in low or 'climbing' in low:
        out['exertional'] = True

    return out

In [39]:
tests = [
    "Patient denies nausea but reports severe chest pressure and shortness of breath.",
    "A 55-year-old male smoker has had heavy pressure in the centre of his chest for 20 minutes. It started while walking and he is sweating.",
    "She can't catch her breath, has a racing heart and feels anxious. No chest pain on exertion.",
]

for t in tests:
    print(t)
    print(" ->", extract(t), "\n")

Patient denies nausea but reports severe chest pressure and shortness of breath.
 -> {'nausea': False, 'chest_pressure': True, 'breathlessness': True} 

A 55-year-old male smoker has had heavy pressure in the centre of his chest for 20 minutes. It started while walking and he is sweating.
 -> {'chest_pressure': True, 'sweating': True, 'age': 55, 'smoking': True, 'sex': 'M', 'duration_min': 20, 'exertional': True} 

She can't catch her breath, has a racing heart and feels anxious. No chest pain on exertion.
 -> {'palpitations': True, 'anxiety': True, 'chest_pain': False, 'exertional': True} 



In [40]:
# What did the raw scispaCy model find, before our synonym + negation layers?
doc_raw = spacy.load('en_core_sci_md')(tests[0])
print([(e.text, e.label_) for e in doc_raw.ents])

[('Patient', 'ENTITY'), ('nausea', 'ENTITY'), ('reports', 'ENTITY'), ('chest pressure', 'ENTITY'), ('shortness of breath', 'ENTITY')]


**Baseline for the comparison experiment (rule-based only, no pretrained model):**

In [41]:
import re
def rule_baseline(text):
    low = text.lower(); out = {}
    for node, phrases in SYNONYMS.items():
        for p in phrases:
            for m in re.finditer(re.escape(p), low):
                window = low[max(0, m.start()-25):m.start()]
                neg = bool(re.search(r'\b(no|denies|denied|without|not)\b', window))
                out[node] = not neg
    return out

for t in tests:
    print('rules :', rule_baseline(t))
    print('model :', {k: v for k, v in extract(t).items() if k in NODES}, '\n')

rules : {'breathlessness': True, 'chest_pressure': True, 'nausea': False}
model : {'nausea': False, 'chest_pressure': True, 'breathlessness': True} 

rules : {'chest_pressure': True, 'sweating': True}
model : {'chest_pressure': True, 'sweating': True} 

rules : {'chest_pain': False, 'palpitations': True, 'anxiety': True}
model : {'palpitations': True, 'anxiety': True, 'chest_pain': False} 



## Part 4: Pre-existing library: pgmpy Bayesian Network (toy version)

In [45]:
try:
    from pgmpy.models import DiscreteBayesianNetwork as BN
except ImportError:
    from pgmpy.models import BayesianNetwork as BN
from pgmpy.factors.discrete import TabularCPD
from pgmpy.inference import VariableElimination

# Condition -> symptoms (naive-Bayes structure). Numbers here are TOY values for the demo only.
model = BN([('Condition', 'Sweating'), ('Condition', 'Exertional'), ('Condition', 'Nausea')])
states = ['ACS', 'GERD', 'Panic']
cpd_c = TabularCPD('Condition', 3, [[0.30], [0.40], [0.30]], state_names={'Condition': states})
def sym(name, p_yes):   # p_yes = P(symptom=yes | ACS, GERD, Panic)
    return TabularCPD(name, 2, [[1-p for p in p_yes], p_yes],
                      evidence=['Condition'], evidence_card=[3],
                      state_names={name: ['no', 'yes'], 'Condition': states})
cpd_s = sym('Sweating',   [0.70, 0.10, 0.40])
cpd_e = sym('Exertional', [0.80, 0.05, 0.10])
cpd_n = sym('Nausea',     [0.50, 0.30, 0.20])
model.add_cpds(cpd_c, cpd_s, cpd_e, cpd_n)
print("Model valid:", model.check_model())

infer = VariableElimination(model)
print("\nPrior:");                      print(infer.query(['Condition']))
print("\nAfter Exertional=yes, Sweating=yes:")
print(infer.query(['Condition'], evidence={'Exertional': 'yes', 'Sweating': 'yes'}))

Model valid: True

Prior:
+------------------+------------------+
| Condition        |   phi(Condition) |
+==================+==================+
| Condition(ACS)   |           0.3000 |
+------------------+------------------+
| Condition(GERD)  |           0.4000 |
+------------------+------------------+
| Condition(Panic) |           0.3000 |
+------------------+------------------+

After Exertional=yes, Sweating=yes:
+------------------+------------------+
| Condition        |   phi(Condition) |
+==================+==================+
| Condition(ACS)   |           0.9231 |
+------------------+------------------+
| Condition(GERD)  |           0.0110 |
+------------------+------------------+
| Condition(Panic) |           0.0659 |
+------------------+------------------+


## Part 5: Active questioning, Expected Information Gain on the toy network
Entropy = uncertainty. EIG of a question = current entropy minus the expected entropy after hearing the answer.

In [46]:
def entropy(p):
    p = np.array(p, dtype=float); p = p[p > 0]
    return float(-(p * np.log2(p)).sum())

def posterior(evidence):
    return infer.query(['Condition'], evidence=evidence, show_progress=False).values

def expected_info_gain(question, evidence):
    h_now = entropy(posterior(evidence))
    pq = infer.query([question], evidence=evidence, show_progress=False)
    exp_h = 0.0
    for i, ans in enumerate(pq.state_names[question]):
        p_ans = pq.values[i]
        if p_ans > 0:
            exp_h += p_ans * entropy(posterior({**evidence, question: ans}))
    return h_now - exp_h

evidence = {'Sweating': 'yes'}        # what we already know
unanswered = [n for n in ['Sweating', 'Exertional', 'Nausea'] if n not in evidence]
scores = {q: round(expected_info_gain(q, evidence), 3) for q in unanswered}
print("Known evidence:", evidence)
print("Posterior:", dict(zip(states, posterior(evidence).round(3))))
print("Entropy now:", round(entropy(posterior(evidence)), 3), "bits")
print("EIG per question:", scores)
print("Ask next ->", max(scores, key=scores.get))

Known evidence: {'Sweating': 'yes'}
Posterior: {'ACS': 0.568, 'GERD': 0.108, 'Panic': 0.324}
Entropy now: 1.338 bits
EIG per question: {'Exertional': 0.407, 'Nausea': 0.062}
Ask next -> Exertional


In [47]:
import plotly.express as px
fig = px.bar(x=list(scores), y=list(scores.values()), labels={'x': 'Candidate question', 'y': 'Expected information gain (bits)'},
             title='Which question should we ask next?')
fig.show()

## Status summary (for the teacher)
- **Done tonight:** dataset loaded and explored, filtered to chest-pain conditions, evidence table built, pretrained biomedical model running with our negation and synonym layers, rule-based baseline for comparison, pgmpy network running, EIG working on a toy network.
- **Next milestone:** replace toy CPTs with CPTs learned from the filtered DDXPlus data, expand to ~15 nodes, connect NLP output to the network, build the Streamlit loop, run the evaluation (NLP vs baseline, EIG vs random vs fixed order).
- **Limitation to state honestly:** DDXPlus is synthetic; toy CPT numbers in Part 4 are placeholders, not clinical values.